# Version 2 DistilBERT Development Training

This notebook fine-tunes the locked DistilBERT challenger on the five fixed
2024 development folds, creates exactly one out-of-fold result for every
development row, applies the precommitted median-best-epoch rule, and trains
one frozen final development model.

Guardrails:

- use only the locked 2024 development partition for tokenization, training,
  checkpoint selection, and OOF evaluation;
- reconstruct the 2024 final internal-test boundary only to verify exclusion;
- do not tokenize or score the final internal test;
- do not access 2025 or 2026 data;
- do not display complaint narratives, complaint IDs, normalized-text hashes,
  row-level labels, logits, predictions, or scores; and
- keep every model, checkpoint, cache, and row-level OOF artifact under the
  Git-ignored `models/v2_distilbert_challenger/` directory.

Valid completed folds are reusable only when their local metadata, source
fingerprint, model/tokenizer revisions, label mapping, configuration, fold
indices, and row counts match the locked experiment.

## 1. Environment, paths, and locked controls

In [1]:
from __future__ import annotations

from pathlib import Path
import gc
import hashlib
import importlib.metadata as package_metadata
import json
import math
import os
import platform
import random
import shutil
import statistics
import subprocess
import sys
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import scipy
import seaborn as sns
import sklearn
import tokenizers
import torch
import transformers
from scipy.special import softmax
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    DataCollatorWithPadding,
    DistilBertForSequenceClassification,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
warnings.filterwarnings(
    "ignore",
    message=r"`huggingface_hub` cache-system uses symlinks by default.*",
    category=UserWarning,
    module=r"huggingface_hub\.file_download",
)


def find_project_root(start_path: Path) -> Path:
    current = start_path.resolve()
    for candidate in [current, *current.parents]:
        expected = candidate / "data" / "processed" / "cfpb_complaints_2024_cleaned.csv"
        if expected.exists() and (candidate / ".gitignore").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository and locked 2024 source.")


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "cfpb_complaints_2024_cleaned.csv"
LOCAL_ROOT = PROJECT_ROOT / "models" / "v2_distilbert_challenger"
HF_CACHE_DIR = LOCAL_ROOT / "hf_cache"
FOLDS_DIR = LOCAL_ROOT / "folds"
OOF_DIR = LOCAL_ROOT / "oof"
FINAL_DIR = LOCAL_ROOT / "final"
FINAL_TEMP_DIR = LOCAL_ROOT / "final_training_tmp"
FIGURE_PATH = PROJECT_ROOT / "reports" / "figures" / "v2_development_confusion_matrix.png"

for local_directory in (LOCAL_ROOT, HF_CACHE_DIR, FOLDS_DIR, OOF_DIR):
    local_directory.mkdir(parents=True, exist_ok=True)
FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)

EXPECTED_ENVIRONMENT = {
    "Python": "3.11.15",
    "Pandas": "3.0.3",
    "NumPy": "2.4.6",
    "SciPy": "1.16.3",
    "Scikit-learn": "1.9.0",
    "Transformers": "4.57.6",
    "Tokenizers": "0.22.2",
    "PyTorch base": "2.9.1",
    "Datasets": "4.4.2",
    "Accelerate": "1.12.0",
    "Evaluate": "0.4.6",
    "Safetensors": "0.7.0",
}
observed_environment = {
    "Python": platform.python_version(),
    "Pandas": pd.__version__,
    "NumPy": np.__version__,
    "SciPy": scipy.__version__,
    "Scikit-learn": sklearn.__version__,
    "Transformers": transformers.__version__,
    "Tokenizers": tokenizers.__version__,
    "PyTorch base": torch.__version__.split("+")[0],
    "Datasets": package_metadata.version("datasets"),
    "Accelerate": package_metadata.version("accelerate"),
    "Evaluate": package_metadata.version("evaluate"),
    "Safetensors": package_metadata.version("safetensors"),
}

EXPECTED_SOURCE_SIZE = 54_908_639
EXPECTED_SOURCE_SHA256 = "b115eb0c4a20a881a6a45bfb74cb7d715a726537372baa7d68f09d657cdfd919"
EXPECTED_SOURCE_ROWS = 50_000
EXPECTED_CONFLICTING_GROUPS = 74
EXPECTED_LOCKED_CONFLICT_ROWS = 1_780
EXPECTED_SAME_LABEL_ROWS_REMOVED = 14_374
EXPECTED_MODELING_ROWS = 33_042
EXPECTED_DEVELOPMENT_ROWS = 26_433
EXPECTED_FINAL_TEST_ROWS = 6_609
OUTER_SPLITS = 5
DEVELOPMENT_SPLITS = 5
FINAL_TEST_FOLD = 0
RANDOM_STATE = 42

CANONICAL_LABELS = (
    "Checking or savings account",
    "Credit card",
    "Credit reporting or other personal consumer reports",
    "Debt collection",
    "Money transfer, virtual currency, or money service",
    "Mortgage",
    "Student loan",
    "Vehicle loan or lease",
)
DISPLAY_LABELS = (
    "Checking / savings",
    "Credit card",
    "Credit reporting",
    "Debt collection",
    "Money transfer",
    "Mortgage",
    "Student loan",
    "Vehicle loan / lease",
)
label2id = {label: label_id for label_id, label in enumerate(CANONICAL_LABELS)}
id2label = {label_id: label for label, label_id in label2id.items()}

MODEL_ID = "distilbert/distilbert-base-uncased"
MODEL_REVISION = "12040accade4e8a0f71eabdb258fecc2e7e948be"
TOKENIZER_ID = MODEL_ID
TOKENIZER_REVISION = MODEL_REVISION
MODEL_CLASS_NAME = "DistilBertForSequenceClassification"
MAX_LENGTH = 256
MAX_EPOCHS = 4

PRIMARY_CONFIG = {
    "name": "primary",
    "train_batch_size": 8,
    "gradient_accumulation_steps": 2,
    "effective_batch_size": 16,
    "eval_batch_size": 16,
}
FALLBACK_CONFIG = {
    "name": "fallback",
    "train_batch_size": 4,
    "gradient_accumulation_steps": 4,
    "effective_batch_size": 16,
    "eval_batch_size": 8,
}
FIXED_TRAINING_SETTINGS = {
    "random_seed": 42,
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    "maximum_fold_epochs": 4,
    "warmup_ratio": 0.10,
    "optimizer": "adamw_torch",
    "adam_beta1": 0.9,
    "adam_beta2": 0.999,
    "adam_epsilon": 1e-8,
    "lr_scheduler_type": "linear",
    "gradient_clipping": 1.0,
    "fp16": True,
    "bf16": False,
    "label_smoothing_factor": 0.0,
    "checkpoint_metric": "macro_f1",
    "early_stopping_patience": 1,
    "early_stopping_threshold": 0.0,
}

conda_environment = Path(sys.prefix).name
git_sha = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip()
git_branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True
).strip()
disk_free_gib = shutil.disk_usage(PROJECT_ROOT).free / (1024**3)

if conda_environment != "complaint-v2":
    raise RuntimeError(f"Wrong Conda environment: {conda_environment}")
if observed_environment != EXPECTED_ENVIRONMENT:
    raise RuntimeError(f"Version 2 environment mismatch: {observed_environment}")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable; locked GPU training cannot start.")
if torch.cuda.get_device_name(0) != "NVIDIA GeForce GTX 1650":
    raise RuntimeError(f"Unexpected GPU: {torch.cuda.get_device_name(0)}")
if disk_free_gib < 10:
    raise RuntimeError(f"Insufficient free disk space: {disk_free_gib:.2f} GiB")
if PRIMARY_CONFIG["effective_batch_size"] != 16:
    raise RuntimeError("Primary effective batch size is not locked at 16.")
if FALLBACK_CONFIG["effective_batch_size"] != 16:
    raise RuntimeError("Fallback effective batch size is not locked at 16.")

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
torch.cuda.manual_seed_all(RANDOM_STATE)
set_seed(RANDOM_STATE)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

driver_version = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
    text=True,
).strip().splitlines()[0]
hardware_record = {
    "Operating system": platform.platform(),
    "CPU": platform.processor(),
    "RAM GiB": round(psutil.virtual_memory().total / (1024**3), 2),
    "GPU": torch.cuda.get_device_name(0),
    "GPU memory MiB": round(torch.cuda.get_device_properties(0).total_memory / (1024**2)),
    "CUDA runtime": torch.version.cuda,
    "NVIDIA driver": driver_version,
}

print("Project root located: PASS")
print(f"Git branch: {git_branch}")
print(f"Baseline Git commit: {git_sha}")
print(f"Conda environment: {conda_environment}")
for package_name, version in observed_environment.items():
    print(f"{package_name}: {version}")
print(f"PyTorch runtime: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {hardware_record['GPU']} ({hardware_record['GPU memory MiB']:,} MiB)")
print(f"CUDA runtime / driver: {hardware_record['CUDA runtime']} / {driver_version}")
print(f"Free repository-drive space: {disk_free_gib:.2f} GiB")
print("Environment and locked controls: PASS")

Project root located: PASS
Git branch: v2/issue-3-distilbert-training
Baseline Git commit: 0ec4759cebc3f823b67e05121cd26a5c4cfc537a
Conda environment: complaint-v2
Python: 3.11.15
Pandas: 3.0.3
NumPy: 2.4.6
SciPy: 1.16.3
Scikit-learn: 1.9.0
Transformers: 4.57.6
Tokenizers: 0.22.2
PyTorch base: 2.9.1
Datasets: 4.4.2
Accelerate: 1.12.0
Evaluate: 0.4.6
Safetensors: 0.7.0
PyTorch runtime: 2.9.1+cu126
CUDA available: True
GPU: NVIDIA GeForce GTX 1650 (4,096 MiB)
CUDA runtime / driver: 12.6 / 591.86
Free repository-drive space: 2709.84 GiB
Environment and locked controls: PASS


## 2. Reconstruct and verify the locked 2024 development data

In [2]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file_handle:
        for chunk in iter(lambda: file_handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def normalize_grouping_text(value: object) -> str:
    normalized = " ".join(str(value).strip().split())
    if not normalized:
        raise ValueError("Normalized grouping text must not be empty.")
    return normalized


def stable_text_hash(value: object) -> str:
    return hashlib.sha256(normalize_grouping_text(value).encode("utf-8")).hexdigest()


if not DATA_PATH.is_file():
    raise FileNotFoundError("Locked 2024 cleaned source is unavailable.")
source_size = DATA_PATH.stat().st_size
source_sha256 = sha256_file(DATA_PATH)
if source_size != EXPECTED_SOURCE_SIZE:
    raise RuntimeError(f"Source-size mismatch: {source_size}")
if source_sha256 != EXPECTED_SOURCE_SHA256:
    raise RuntimeError("Source SHA-256 does not match the locked fingerprint.")

required_columns = ["clean_complaint_text", "product"]
source_df = pd.read_csv(DATA_PATH, usecols=required_columns)
accessed_data_paths = {DATA_PATH.resolve()}
if len(source_df) != EXPECTED_SOURCE_ROWS:
    raise RuntimeError(f"Source-row mismatch: {len(source_df)}")
if set(source_df.columns) != set(required_columns):
    raise RuntimeError(f"Required-column mismatch: {list(source_df.columns)}")
if source_df[required_columns].isna().any().any():
    raise RuntimeError("Required columns contain missing values.")
if source_df["clean_complaint_text"].astype(str).str.strip().eq("").any():
    raise RuntimeError("Blank cleaned narratives are present.")
if source_df["product"].astype(str).str.strip().eq("").any():
    raise RuntimeError("Blank product labels are present.")

working_df = source_df[required_columns].copy()
working_df["source_row_order"] = np.arange(len(working_df), dtype=np.int64)
working_df["normalized_text_hash"] = working_df["clean_complaint_text"].map(stable_text_hash)

group_label_counts = working_df.groupby("normalized_text_hash", sort=False)["product"].nunique()
conflicting_hashes = set(group_label_counts[group_label_counts > 1].index)
conflicting_label_groups = len(conflicting_hashes)

locked_scope_df = working_df[working_df["product"].isin(CANONICAL_LABELS)].copy()
locked_conflict_mask = locked_scope_df["normalized_text_hash"].isin(conflicting_hashes)
locked_scope_conflicting_rows = int(locked_conflict_mask.sum())
scope_without_conflicts = locked_scope_df.loc[~locked_conflict_mask].copy()
remediated_df = scope_without_conflicts.drop_duplicates(
    subset=["normalized_text_hash", "product"],
    keep="first",
).reset_index(drop=True)
same_label_rows_removed = len(scope_without_conflicts) - len(remediated_df)

assert conflicting_label_groups == EXPECTED_CONFLICTING_GROUPS
assert locked_scope_conflicting_rows == EXPECTED_LOCKED_CONFLICT_ROWS
assert same_label_rows_removed == EXPECTED_SAME_LABEL_ROWS_REMOVED
assert len(remediated_df) == EXPECTED_MODELING_ROWS
assert remediated_df["normalized_text_hash"].is_unique
assert remediated_df["source_row_order"].is_monotonic_increasing
assert set(remediated_df["product"]) == set(CANONICAL_LABELS)

outer_splitter = StratifiedGroupKFold(
    n_splits=OUTER_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)
outer_folds = list(
    outer_splitter.split(
        remediated_df["clean_complaint_text"],
        remediated_df["product"],
        groups=remediated_df["normalized_text_hash"],
    )
)
development_indices, final_test_indices = outer_folds[FINAL_TEST_FOLD]
development_df = remediated_df.iloc[development_indices].reset_index(drop=True)
final_test_reference = remediated_df.iloc[final_test_indices]

development_final_overlap = len(
    set(development_df["normalized_text_hash"]).intersection(
        set(final_test_reference["normalized_text_hash"])
    )
)
assert len(development_df) == EXPECTED_DEVELOPMENT_ROWS
assert len(final_test_reference) == EXPECTED_FINAL_TEST_ROWS
assert development_final_overlap == 0
assert set(development_df["product"]) == set(CANONICAL_LABELS)
assert set(final_test_reference["product"]) == set(CANONICAL_LABELS)

development_splitter = StratifiedGroupKFold(
    n_splits=DEVELOPMENT_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)
development_folds = list(
    development_splitter.split(
        development_df["clean_complaint_text"],
        development_df["product"],
        groups=development_df["normalized_text_hash"],
    )
)

fold_rows = []
validation_assignments = np.zeros(len(development_df), dtype=np.int8)
for fold_number, (fold_train_indices, fold_validation_indices) in enumerate(development_folds):
    train_groups = set(development_df.iloc[fold_train_indices]["normalized_text_hash"])
    validation_groups = set(development_df.iloc[fold_validation_indices]["normalized_text_hash"])
    fold_overlap = len(train_groups.intersection(validation_groups))
    validation_assignments[fold_validation_indices] += 1
    train_class_count = int(development_df.iloc[fold_train_indices]["product"].nunique())
    validation_class_count = int(
        development_df.iloc[fold_validation_indices]["product"].nunique()
    )
    assert fold_overlap == 0
    assert train_class_count == len(CANONICAL_LABELS)
    assert validation_class_count == len(CANONICAL_LABELS)
    fold_rows.append(
        {
            "Fold": fold_number,
            "Training rows": len(fold_train_indices),
            "Validation rows": len(fold_validation_indices),
            "Group overlap": fold_overlap,
            "Training classes": train_class_count,
            "Validation classes": validation_class_count,
        }
    )

assert len(development_folds) == DEVELOPMENT_SPLITS
assert np.all(validation_assignments == 1)

development_label_ids = development_df["product"].map(label2id).to_numpy(dtype=np.int64)
development_class_support = np.bincount(
    development_label_ids, minlength=len(CANONICAL_LABELS)
)
assert len(development_label_ids) == EXPECTED_DEVELOPMENT_ROWS
assert np.all(development_class_support > 0)

print("Locked 2024 reconstruction:")
print(f"- Source: data/processed/cfpb_complaints_2024_cleaned.csv")
print(f"- File size: {source_size:,} bytes (PASS)")
print("- SHA-256: matches the committed fingerprint (PASS)")
print(f"- Source rows: {len(source_df):,}")
print(f"- Conflicting-label groups: {conflicting_label_groups:,}")
print(f"- Locked-scope conflicting rows excluded: {locked_scope_conflicting_rows:,}")
print(f"- Repeated same-label rows removed: {same_label_rows_removed:,}")
print(f"- Corrected modeling rows: {len(remediated_df):,}")
print(f"- Development rows: {len(development_df):,}")
print(f"- Final internal-test boundary rows: {len(final_test_reference):,}")
print(f"- Development/final-test normalized-text overlap: {development_final_overlap}")
print(pd.DataFrame(fold_rows).to_string(index=False))
print("Every development row validates exactly once: PASS")
print("The final internal-test boundary was reconstructed only to verify exclusion.")

Locked 2024 reconstruction:
- Source: data/processed/cfpb_complaints_2024_cleaned.csv
- File size: 54,908,639 bytes (PASS)
- SHA-256: matches the committed fingerprint (PASS)
- Source rows: 50,000
- Conflicting-label groups: 74
- Locked-scope conflicting rows excluded: 1,780
- Repeated same-label rows removed: 14,374
- Corrected modeling rows: 33,042
- Development rows: 26,433
- Final internal-test boundary rows: 6,609
- Development/final-test normalized-text overlap: 0
 Fold  Training rows  Validation rows  Group overlap  Training classes  Validation classes
    0          21146             5287              0                 8                   8
    1          21146             5287              0                 8                   8
    2          21146             5287              0                 8                   8
    3          21147             5286              0                 8                   8
    4          21147             5286              0                 8

## 3. Development-only tokenization and dynamic padding

In [3]:
tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_ID,
    revision=TOKENIZER_REVISION,
    cache_dir=HF_CACHE_DIR,
    use_fast=True,
)
if tokenizer.__class__.__name__ != "DistilBertTokenizerFast":
    raise RuntimeError(f"Unexpected tokenizer class: {tokenizer.__class__.__name__}")
if tokenizer.vocab_size != 30_522 or tokenizer.model_max_length != 512:
    raise RuntimeError("Tokenizer metadata does not match the locked revision.")

tokenized_input_ids = []
tokenized_attention_masks = []
tokenization_batch_size = 256
for start in range(0, len(development_df), tokenization_batch_size):
    stop = min(start + tokenization_batch_size, len(development_df))
    encoded = tokenizer(
        development_df["clean_complaint_text"].iloc[start:stop].tolist(),
        add_special_tokens=True,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
        return_attention_mask=True,
        return_token_type_ids=False,
        verbose=False,
    )
    tokenized_input_ids.extend(encoded["input_ids"])
    tokenized_attention_masks.extend(encoded["attention_mask"])
    del encoded

assert len(tokenized_input_ids) == EXPECTED_DEVELOPMENT_ROWS
assert len(tokenized_attention_masks) == EXPECTED_DEVELOPMENT_ROWS
assert max(map(len, tokenized_input_ids)) <= MAX_LENGTH
assert min(map(len, tokenized_input_ids)) >= 2


class EncodedDevelopmentDataset(Dataset):
    def __init__(self, positions):
        self.positions = np.asarray(positions, dtype=np.int64)

    def __len__(self):
        return len(self.positions)

    def __getitem__(self, item):
        position = int(self.positions[item])
        return {
            "input_ids": tokenized_input_ids[position],
            "attention_mask": tokenized_attention_masks[position],
            "labels": int(development_label_ids[position]),
        }


data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding="longest",
    return_tensors="pt",
)
smoke_positions = np.array([0, len(development_label_ids) // 3, len(development_label_ids) - 1])
smoke_features = [EncodedDevelopmentDataset(smoke_positions)[i] for i in range(3)]
smoke_batch = data_collator(smoke_features)
assert smoke_batch["input_ids"].shape == smoke_batch["attention_mask"].shape
assert smoke_batch["input_ids"].shape[0] == 3
assert smoke_batch["input_ids"].shape[1] <= MAX_LENGTH

# Narratives and normalized-text hashes are no longer needed after development tokenization.
del smoke_features, smoke_batch
del source_df, working_df, locked_scope_df, scope_without_conflicts
del remediated_df, final_test_reference, development_df
gc.collect()

print("Development-only tokenization:")
print(f"- Rows tokenized: {len(tokenized_input_ids):,}")
print(f"- Locked maximum length: {MAX_LENGTH}")
print("- Truncation: enabled at the locked maximum")
print("- Padding: dynamic, longest sequence in each batch")
print("- Final internal-test rows tokenized: 0")
print("- 2025/2026 rows accessed: 0")
print("Development tokenization and dynamic padding: PASS")

Development-only tokenization:
- Rows tokenized: 26,433
- Locked maximum length: 256
- Truncation: enabled at the locked maximum
- Padding: dynamic, longest sequence in each batch
- Final internal-test rows tokenized: 0
- 2025/2026 rows accessed: 0
Development tokenization and dynamic padding: PASS


## 4. Training, metrics, and validated local resumability

In [4]:
def canonical_json(value) -> str:
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def sha256_array(values: np.ndarray) -> str:
    array = np.ascontiguousarray(values)
    return hashlib.sha256(array.view(np.uint8)).hexdigest()


def safe_remove_directory(path: Path) -> None:
    resolved_path = path.resolve()
    resolved_root = LOCAL_ROOT.resolve()
    if resolved_path == resolved_root or resolved_root not in resolved_path.parents:
        raise RuntimeError(f"Refusing to remove path outside the local artifact root: {path}")
    if path.exists():
        shutil.rmtree(path)


def clean_cuda() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def classification_metrics(true_labels: np.ndarray, predicted_labels: np.ndarray) -> dict:
    macro = precision_recall_fscore_support(
        true_labels,
        predicted_labels,
        labels=np.arange(len(CANONICAL_LABELS)),
        average="macro",
        zero_division=0,
    )
    weighted = precision_recall_fscore_support(
        true_labels,
        predicted_labels,
        labels=np.arange(len(CANONICAL_LABELS)),
        average="weighted",
        zero_division=0,
    )
    return {
        "accuracy": float(accuracy_score(true_labels, predicted_labels)),
        "macro_precision": float(macro[0]),
        "macro_recall": float(macro[1]),
        "macro_f1": float(macro[2]),
        "weighted_precision": float(weighted[0]),
        "weighted_recall": float(weighted[1]),
        "weighted_f1": float(weighted[2]),
    }


def trainer_compute_metrics(eval_prediction) -> dict:
    logits = eval_prediction.predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    predictions = np.argmax(logits, axis=1)
    metrics = classification_metrics(
        np.asarray(eval_prediction.label_ids),
        predictions,
    )
    return {
        "accuracy": metrics["accuracy"],
        "macro_f1": metrics["macro_f1"],
        "weighted_f1": metrics["weighted_f1"],
    }


def initialize_fresh_model():
    return DistilBertForSequenceClassification.from_pretrained(
        MODEL_ID,
        revision=MODEL_REVISION,
        cache_dir=HF_CACHE_DIR,
        num_labels=len(CANONICAL_LABELS),
        label2id=label2id,
        id2label=id2label,
        ignore_mismatched_sizes=True,
    )


def build_fold_training_arguments(output_dir: Path, config: dict, fold_number: int):
    return TrainingArguments(
        output_dir=str(output_dir),
        overwrite_output_dir=True,
        run_name=f"v2-distilbert-fold-{fold_number}",
        seed=RANDOM_STATE,
        data_seed=RANDOM_STATE,
        learning_rate=FIXED_TRAINING_SETTINGS["learning_rate"],
        weight_decay=FIXED_TRAINING_SETTINGS["weight_decay"],
        num_train_epochs=MAX_EPOCHS,
        per_device_train_batch_size=config["train_batch_size"],
        gradient_accumulation_steps=config["gradient_accumulation_steps"],
        per_device_eval_batch_size=config["eval_batch_size"],
        warmup_ratio=FIXED_TRAINING_SETTINGS["warmup_ratio"],
        lr_scheduler_type=FIXED_TRAINING_SETTINGS["lr_scheduler_type"],
        optim=FIXED_TRAINING_SETTINGS["optimizer"],
        adam_beta1=FIXED_TRAINING_SETTINGS["adam_beta1"],
        adam_beta2=FIXED_TRAINING_SETTINGS["adam_beta2"],
        adam_epsilon=FIXED_TRAINING_SETTINGS["adam_epsilon"],
        max_grad_norm=FIXED_TRAINING_SETTINGS["gradient_clipping"],
        fp16=True,
        bf16=False,
        label_smoothing_factor=0.0,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=1,
        save_only_model=True,
        report_to=[],
        disable_tqdm=True,
        dataloader_num_workers=0,
        dataloader_pin_memory=True,
        eval_accumulation_steps=8,
        remove_unused_columns=False,
    )


def fold_signature(
    fold_number: int,
    train_indices: np.ndarray,
    validation_indices: np.ndarray,
    config: dict,
) -> dict:
    return {
        "source_sha256": EXPECTED_SOURCE_SHA256,
        "source_size": EXPECTED_SOURCE_SIZE,
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "tokenizer_id": TOKENIZER_ID,
        "tokenizer_revision": TOKENIZER_REVISION,
        "model_class": MODEL_CLASS_NAME,
        "fold_number": int(fold_number),
        "maximum_length": MAX_LENGTH,
        "label2id": label2id,
        "training_settings": FIXED_TRAINING_SETTINGS,
        "batch_configuration": config,
        "training_rows": int(len(train_indices)),
        "validation_rows": int(len(validation_indices)),
        "training_indices_sha256": sha256_array(np.asarray(train_indices, dtype=np.int64)),
        "validation_indices_sha256": sha256_array(
            np.asarray(validation_indices, dtype=np.int64)
        ),
    }


def load_valid_completed_fold(
    fold_number: int,
    train_indices: np.ndarray,
    validation_indices: np.ndarray,
):
    fold_dir = FOLDS_DIR / f"fold_{fold_number}"
    metadata_path = fold_dir / "completed_metadata.json"
    result_path = fold_dir / "validation_outputs.npz"
    if not metadata_path.is_file() or not result_path.is_file():
        return None
    try:
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        valid_signatures = [
            fold_signature(fold_number, train_indices, validation_indices, PRIMARY_CONFIG),
            fold_signature(fold_number, train_indices, validation_indices, FALLBACK_CONFIG),
        ]
        if metadata.get("signature") not in valid_signatures:
            return None
        if metadata.get("result_sha256") != sha256_file(result_path):
            return None
        with np.load(result_path, allow_pickle=False) as payload:
            saved_validation_indices = payload["validation_indices"]
            true_labels = payload["true_labels"]
            logits = payload["logits"]
            predictions = payload["predictions"]
        if not np.array_equal(saved_validation_indices, validation_indices):
            return None
        if not np.array_equal(true_labels, development_label_ids[validation_indices]):
            return None
        if logits.shape != (len(validation_indices), len(CANONICAL_LABELS)):
            return None
        if predictions.shape != (len(validation_indices),):
            return None
        if not np.isfinite(logits).all():
            return None
        if not np.array_equal(predictions, np.argmax(logits, axis=1)):
            return None
        return {
            "metadata": metadata,
            "validation_indices": saved_validation_indices,
            "true_labels": true_labels,
            "logits": logits,
            "predictions": predictions,
            "reused": True,
        }
    except (OSError, ValueError, KeyError, json.JSONDecodeError):
        return None


def identify_best_epoch(log_history: list[dict], best_metric: float) -> int:
    matching = [
        entry
        for entry in log_history
        if "eval_macro_f1" in entry
        and "epoch" in entry
        and math.isclose(
            float(entry["eval_macro_f1"]),
            float(best_metric),
            rel_tol=0.0,
            abs_tol=1e-12,
        )
    ]
    if not matching:
        raise RuntimeError("Could not map the best validation Macro F1 to an epoch.")
    best_epoch = int(round(float(matching[0]["epoch"])))
    if best_epoch < 1 or best_epoch > MAX_EPOCHS:
        raise RuntimeError(f"Invalid best epoch: {best_epoch}")
    return best_epoch


def execute_fold(
    fold_number: int,
    train_indices: np.ndarray,
    validation_indices: np.ndarray,
    config: dict,
):
    fold_dir = FOLDS_DIR / f"fold_{fold_number}"
    checkpoint_dir = fold_dir / "checkpoints"
    if fold_dir.exists():
        safe_remove_directory(fold_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    set_seed(RANDOM_STATE)
    train_dataset = EncodedDevelopmentDataset(train_indices)
    validation_dataset = EncodedDevelopmentDataset(validation_indices)
    model = initialize_fresh_model()
    if model.__class__.__name__ != MODEL_CLASS_NAME or model.num_labels != 8:
        raise RuntimeError("Fresh fold model does not match the locked architecture.")

    trainer = Trainer(
        model=model,
        args=build_fold_training_arguments(checkpoint_dir, config, fold_number),
        train_dataset=train_dataset,
        eval_dataset=validation_dataset,
        data_collator=data_collator,
        compute_metrics=trainer_compute_metrics,
        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=FIXED_TRAINING_SETTINGS[
                    "early_stopping_patience"
                ],
                early_stopping_threshold=FIXED_TRAINING_SETTINGS[
                    "early_stopping_threshold"
                ],
            )
        ],
    )

    clean_cuda()
    torch.cuda.reset_peak_memory_stats()
    wall_start = time.perf_counter()
    train_output = trainer.train()
    training_wall_seconds = time.perf_counter() - wall_start

    prediction_output = trainer.predict(validation_dataset)
    logits = prediction_output.predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    logits = np.asarray(logits, dtype=np.float32)
    true_labels = np.asarray(prediction_output.label_ids, dtype=np.int64)
    predictions = np.argmax(logits, axis=1).astype(np.int64)
    metrics = classification_metrics(true_labels, predictions)
    best_metric = float(trainer.state.best_metric)
    best_epoch = identify_best_epoch(trainer.state.log_history, best_metric)
    peak_allocated_mib = torch.cuda.max_memory_allocated() / (1024**2)
    peak_reserved_mib = torch.cuda.max_memory_reserved() / (1024**2)

    result_path = fold_dir / "validation_outputs.npz"
    np.savez_compressed(
        result_path,
        validation_indices=np.asarray(validation_indices, dtype=np.int64),
        true_labels=true_labels,
        logits=logits,
        predictions=predictions,
    )
    metadata = {
        "signature": fold_signature(
            fold_number, train_indices, validation_indices, config
        ),
        "best_epoch": best_epoch,
        "best_validation_macro_f1": best_metric,
        "metrics": metrics,
        "training_runtime_seconds": float(
            train_output.metrics.get("train_runtime", training_wall_seconds)
        ),
        "training_wall_seconds": float(training_wall_seconds),
        "peak_allocated_mib": float(peak_allocated_mib),
        "peak_reserved_mib": float(peak_reserved_mib),
        "result_path": result_path.relative_to(PROJECT_ROOT).as_posix(),
        "result_size_bytes": result_path.stat().st_size,
        "result_sha256": sha256_file(result_path),
    }
    (fold_dir / "completed_metadata.json").write_text(
        json.dumps(metadata, indent=2, sort_keys=True),
        encoding="utf-8",
    )

    del trainer, model, train_dataset, validation_dataset, prediction_output, train_output
    clean_cuda()
    if checkpoint_dir.exists():
        safe_remove_directory(checkpoint_dir)

    return {
        "metadata": metadata,
        "validation_indices": np.asarray(validation_indices, dtype=np.int64),
        "true_labels": true_labels,
        "logits": logits,
        "predictions": predictions,
        "reused": False,
    }


def is_cuda_oom(error: BaseException) -> bool:
    return isinstance(error, torch.OutOfMemoryError) or (
        isinstance(error, RuntimeError)
        and "out of memory" in str(error).lower()
        and "cuda" in str(error).lower()
    )


print("Training helpers and strict local-resume validation: READY")
print("External reporting integrations: DISABLED")
print("Checkpoint metric: validation Macro F1")
print("Primary batch configuration:", PRIMARY_CONFIG)
print("Permitted OOM fallback:", FALLBACK_CONFIG)

Training helpers and strict local-resume validation: READY
External reporting integrations: DISABLED
Checkpoint metric: validation Macro F1
Primary batch configuration: {'name': 'primary', 'train_batch_size': 8, 'gradient_accumulation_steps': 2, 'effective_batch_size': 16, 'eval_batch_size': 16}
Permitted OOM fallback: {'name': 'fallback', 'train_batch_size': 4, 'gradient_accumulation_steps': 4, 'effective_batch_size': 16, 'eval_batch_size': 8}


## 5. Five-fold training and complete development OOF outputs

In [5]:
oom_state_path = LOCAL_ROOT / "documented_oom_fallback.json"
active_config = PRIMARY_CONFIG
if oom_state_path.is_file():
    oom_state = json.loads(oom_state_path.read_text(encoding="utf-8"))
    if oom_state.get("fallback_configuration") != FALLBACK_CONFIG:
        raise RuntimeError("Existing OOM fallback state does not match the locked fallback.")
    active_config = FALLBACK_CONFIG
    print("Documented prior real-training OOM found; locked fallback remains active.")

fold_results = []
for fold_number, (train_indices, validation_indices) in enumerate(development_folds):
    train_indices = np.asarray(train_indices, dtype=np.int64)
    validation_indices = np.asarray(validation_indices, dtype=np.int64)
    completed = load_valid_completed_fold(
        fold_number, train_indices, validation_indices
    )
    if completed is not None:
        completed_config = completed["metadata"]["signature"]["batch_configuration"]
        if completed_config == FALLBACK_CONFIG:
            active_config = FALLBACK_CONFIG
        fold_results.append(completed)
        print(
            f"Fold {fold_number}: reused validated completed local result "
            f"({completed_config['name']} configuration)."
        )
        continue

    print(
        f"Fold {fold_number}: training from the locked base revision "
        f"with the {active_config['name']} configuration."
    )
    try:
        result = execute_fold(
            fold_number, train_indices, validation_indices, active_config
        )
    except BaseException as error:
        if not is_cuda_oom(error):
            raise
        clean_cuda()
        if active_config == FALLBACK_CONFIG:
            raise RuntimeError(
                f"Fold {fold_number} also failed with the only permitted fallback."
            ) from error
        fold_dir = FOLDS_DIR / f"fold_{fold_number}"
        if fold_dir.exists():
            safe_remove_directory(fold_dir)
        oom_record = {
            "failed_fold": fold_number,
            "primary_configuration": PRIMARY_CONFIG,
            "fallback_configuration": FALLBACK_CONFIG,
            "maximum_length": MAX_LENGTH,
            "effective_batch_size": 16,
            "event": "CUDA out-of-memory during real fold training before completion",
        }
        oom_state_path.write_text(
            json.dumps(oom_record, indent=2, sort_keys=True),
            encoding="utf-8",
        )
        active_config = FALLBACK_CONFIG
        warnings.warn(
            f"Documented CUDA OOM in fold {fold_number}; restarting the fold "
            "with the only permitted fallback.",
            RuntimeWarning,
        )
        result = execute_fold(
            fold_number, train_indices, validation_indices, active_config
        )
    fold_results.append(result)
    fold_metadata = result["metadata"]
    print(
        f"Fold {fold_number}: complete; best epoch {fold_metadata['best_epoch']}; "
        f"Accuracy {fold_metadata['metrics']['accuracy']:.4f}; "
        f"Macro F1 {fold_metadata['metrics']['macro_f1']:.4f}; "
        f"Weighted F1 {fold_metadata['metrics']['weighted_f1']:.4f}."
    )

if len(fold_results) != DEVELOPMENT_SPLITS:
    raise RuntimeError("Not all five folds completed.")

oof_logits = np.full(
    (EXPECTED_DEVELOPMENT_ROWS, len(CANONICAL_LABELS)),
    np.nan,
    dtype=np.float32,
)
oof_predictions = np.full(EXPECTED_DEVELOPMENT_ROWS, -1, dtype=np.int64)
oof_fold_assignments = np.full(EXPECTED_DEVELOPMENT_ROWS, -1, dtype=np.int8)
oof_assignment_counts = np.zeros(EXPECTED_DEVELOPMENT_ROWS, dtype=np.int8)

fold_summary_rows = []
for fold_number, result in enumerate(fold_results):
    validation_indices = result["validation_indices"]
    metadata = result["metadata"]
    oof_logits[validation_indices] = result["logits"]
    oof_predictions[validation_indices] = result["predictions"]
    oof_fold_assignments[validation_indices] = fold_number
    oof_assignment_counts[validation_indices] += 1
    fold_summary_rows.append(
        {
            "Fold": fold_number,
            "Train rows": metadata["signature"]["training_rows"],
            "Validation rows": metadata["signature"]["validation_rows"],
            "Best epoch": metadata["best_epoch"],
            "Accuracy": metadata["metrics"]["accuracy"],
            "Macro F1": metadata["metrics"]["macro_f1"],
            "Weighted F1": metadata["metrics"]["weighted_f1"],
            "Training minutes": metadata["training_runtime_seconds"] / 60.0,
            "Peak allocated MiB": metadata["peak_allocated_mib"],
            "Peak reserved MiB": metadata["peak_reserved_mib"],
            "Configuration": metadata["signature"]["batch_configuration"]["name"],
            "Reused": result["reused"],
        }
    )

assert np.all(oof_assignment_counts == 1)
assert np.all(oof_predictions >= 0)
assert np.isfinite(oof_logits).all()
assert np.array_equal(oof_predictions, np.argmax(oof_logits, axis=1))
assert np.array_equal(np.sort(np.unique(oof_fold_assignments)), np.arange(5))

oof_softmax = softmax(oof_logits.astype(np.float64), axis=1)
sorted_oof_scores = np.sort(oof_softmax, axis=1)
oof_top_scores = sorted_oof_scores[:, -1].astype(np.float32)
oof_score_margins = (sorted_oof_scores[:, -1] - sorted_oof_scores[:, -2]).astype(
    np.float32
)
del oof_softmax, sorted_oof_scores

oof_artifact_path = OOF_DIR / "development_oof_outputs.npz"
np.savez_compressed(
    oof_artifact_path,
    true_labels=development_label_ids,
    predicted_labels=oof_predictions,
    logits=oof_logits,
    fold_assignment=oof_fold_assignments,
    top_softmax_score=oof_top_scores,
    top_two_softmax_margin=oof_score_margins,
)
oof_artifact_size = oof_artifact_path.stat().st_size
oof_artifact_sha256 = sha256_file(oof_artifact_path)

aggregate_oof_metrics = classification_metrics(
    development_label_ids, oof_predictions
)
per_class_precision, per_class_recall, per_class_f1, per_class_support = (
    precision_recall_fscore_support(
        development_label_ids,
        oof_predictions,
        labels=np.arange(len(CANONICAL_LABELS)),
        average=None,
        zero_division=0,
    )
)
oof_confusion_counts = confusion_matrix(
    development_label_ids,
    oof_predictions,
    labels=np.arange(len(CANONICAL_LABELS)),
)
oof_confusion_normalized = oof_confusion_counts / oof_confusion_counts.sum(
    axis=1, keepdims=True
)

per_category_oof = pd.DataFrame(
    {
        "Category": CANONICAL_LABELS,
        "Precision": per_class_precision,
        "Recall": per_class_recall,
        "F1": per_class_f1,
        "Support": per_class_support.astype(int),
    }
)
fold_summary = pd.DataFrame(fold_summary_rows)

oof_local_manifest = {
    "path": oof_artifact_path.relative_to(PROJECT_ROOT).as_posix(),
    "size_bytes": oof_artifact_size,
    "sha256": oof_artifact_sha256,
    "rows": EXPECTED_DEVELOPMENT_ROWS,
    "classes": len(CANONICAL_LABELS),
    "source_sha256": EXPECTED_SOURCE_SHA256,
    "model_revision": MODEL_REVISION,
    "tokenizer_revision": TOKENIZER_REVISION,
    "maximum_length": MAX_LENGTH,
    "label2id": label2id,
}
(OOF_DIR / "development_oof_manifest.json").write_text(
    json.dumps(oof_local_manifest, indent=2, sort_keys=True),
    encoding="utf-8",
)

print("Five-fold results:")
print(
    fold_summary.to_string(
        index=False,
        formatters={
            "Accuracy": "{:.4f}".format,
            "Macro F1": "{:.4f}".format,
            "Weighted F1": "{:.4f}".format,
            "Training minutes": "{:.2f}".format,
            "Peak allocated MiB": "{:.1f}".format,
            "Peak reserved MiB": "{:.1f}".format,
        },
    )
)
print("Aggregate development OOF metrics:")
for metric_name, metric_value in aggregate_oof_metrics.items():
    print(f"- {metric_name}: {metric_value:.6f}")
print("Per-category development OOF metrics:")
print(
    per_category_oof.to_string(
        index=False,
        formatters={
            "Precision": "{:.6f}".format,
            "Recall": "{:.6f}".format,
            "F1": "{:.6f}".format,
        },
    )
)
print(f"Complete OOF assignment: {int(oof_assignment_counts.sum()):,} assignments")
print(f"OOF local artifact: {oof_artifact_path.relative_to(PROJECT_ROOT).as_posix()}")
print(f"OOF artifact size: {oof_artifact_size:,} bytes")
print(f"OOF artifact SHA-256: {oof_artifact_sha256}")
print("Row-level OOF contents remain local and are not displayed.")

Fold 0: reused validated completed local result (primary configuration).
Fold 1: reused validated completed local result (primary configuration).
Fold 2: reused validated completed local result (primary configuration).
Fold 3: reused validated completed local result (primary configuration).
Fold 4: reused validated completed local result (primary configuration).


Five-fold results:
 Fold  Train rows  Validation rows  Best epoch Accuracy Macro F1 Weighted F1 Training minutes Peak allocated MiB Peak reserved MiB Configuration  Reused
    0       21146             5287           4   0.8837   0.7868      0.8819           302.49             1587.9            1746.0       primary    True
    1       21146             5287           2   0.8795   0.7809      0.8771           227.68             1583.5            1744.0       primary    True
    2       21146             5287           4   0.8827   0.7832      0.8806           301.07             1584.3            1748.0       primary    True
    3       21147             5286           4   0.8727   0.7697      0.8710           298.40             1583.8            1748.0       primary    True
    4       21147             5286           4   0.8801   0.7881      0.8773           301.35             1585.9            1758.0       primary    True
Aggregate development OOF metrics:
- accuracy: 0.879734
- macro

## 6. Development OOF confusion matrix

In [6]:
sns.set_theme(style="white")
figure, axis = plt.subplots(figsize=(12, 10))
sns.heatmap(
    oof_confusion_normalized,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    vmin=0,
    vmax=1,
    xticklabels=DISPLAY_LABELS,
    yticklabels=DISPLAY_LABELS,
    cbar_kws={"label": "Row-normalized share"},
    square=True,
    ax=axis,
)
axis.set_title(
    "Version 2 DistilBERT Development OOF Confusion Matrix\n"
    "Five fixed folds; row-normalized",
    pad=16,
)
axis.set_xlabel("Predicted product category")
axis.set_ylabel("Actual product category")
axis.tick_params(axis="x", rotation=45)
axis.tick_params(axis="y", rotation=0)
figure.tight_layout()
figure.savefig(FIGURE_PATH, dpi=200, bbox_inches="tight")
plt.close(figure)

if not FIGURE_PATH.is_file() or FIGURE_PATH.stat().st_size == 0:
    raise RuntimeError("Development confusion-matrix figure was not created.")

print(
    "Saved aggregate figure: "
    f"{FIGURE_PATH.relative_to(PROJECT_ROOT).as_posix()}"
)
print(f"Figure size: {FIGURE_PATH.stat().st_size:,} bytes")
print("Figure contains aggregate row-normalized values only.")

Saved aggregate figure: reports/figures/v2_development_confusion_matrix.png
Figure size: 258,188 bytes
Figure contains aggregate row-normalized values only.


## 7. Median-best-epoch rule and frozen final development model

In [7]:
best_epochs = [int(result["metadata"]["best_epoch"]) for result in fold_results]
selected_final_epoch = int(statistics.median(best_epochs))
if selected_final_epoch < 1 or selected_final_epoch > MAX_EPOCHS:
    raise RuntimeError(f"Invalid median best epoch: {selected_final_epoch}")

print(f"Five fold-best integer epochs: {best_epochs}")
print(f"Precommitted median-best-epoch selection: {selected_final_epoch}")
print("The final epoch was recorded before final model training.")


def final_signature(config: dict) -> dict:
    return {
        "source_sha256": EXPECTED_SOURCE_SHA256,
        "source_size": EXPECTED_SOURCE_SIZE,
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "tokenizer_id": TOKENIZER_ID,
        "tokenizer_revision": TOKENIZER_REVISION,
        "model_class": MODEL_CLASS_NAME,
        "maximum_length": MAX_LENGTH,
        "label2id": label2id,
        "training_settings": FIXED_TRAINING_SETTINGS,
        "batch_configuration": config,
        "development_rows": EXPECTED_DEVELOPMENT_ROWS,
        "fold_best_epochs": best_epochs,
        "selected_final_epoch": selected_final_epoch,
    }


def directory_inventory(directory: Path, excluded_names: set[str] | None = None):
    excluded_names = excluded_names or set()
    records = []
    for file_path in sorted(path for path in directory.rglob("*") if path.is_file()):
        if file_path.name in excluded_names:
            continue
        records.append(
            {
                "path": file_path.relative_to(PROJECT_ROOT).as_posix(),
                "size_bytes": file_path.stat().st_size,
                "sha256": sha256_file(file_path),
            }
        )
    return records


def validate_core_inventory(records: list[dict]) -> bool:
    for record in records:
        file_path = PROJECT_ROOT / record["path"]
        if not file_path.is_file():
            return False
        if file_path.stat().st_size != record["size_bytes"]:
            return False
        if sha256_file(file_path) != record["sha256"]:
            return False
    return bool(records)


def build_final_training_arguments(output_dir: Path, config: dict):
    return TrainingArguments(
        output_dir=str(output_dir),
        overwrite_output_dir=True,
        run_name="v2-distilbert-final-development",
        seed=RANDOM_STATE,
        data_seed=RANDOM_STATE,
        learning_rate=FIXED_TRAINING_SETTINGS["learning_rate"],
        weight_decay=FIXED_TRAINING_SETTINGS["weight_decay"],
        num_train_epochs=selected_final_epoch,
        per_device_train_batch_size=config["train_batch_size"],
        gradient_accumulation_steps=config["gradient_accumulation_steps"],
        per_device_eval_batch_size=config["eval_batch_size"],
        warmup_ratio=FIXED_TRAINING_SETTINGS["warmup_ratio"],
        lr_scheduler_type=FIXED_TRAINING_SETTINGS["lr_scheduler_type"],
        optim=FIXED_TRAINING_SETTINGS["optimizer"],
        adam_beta1=FIXED_TRAINING_SETTINGS["adam_beta1"],
        adam_beta2=FIXED_TRAINING_SETTINGS["adam_beta2"],
        adam_epsilon=FIXED_TRAINING_SETTINGS["adam_epsilon"],
        max_grad_norm=FIXED_TRAINING_SETTINGS["gradient_clipping"],
        fp16=True,
        bf16=False,
        label_smoothing_factor=0.0,
        eval_strategy="no",
        save_strategy="no",
        logging_strategy="epoch",
        load_best_model_at_end=False,
        report_to=[],
        disable_tqdm=True,
        dataloader_num_workers=0,
        dataloader_pin_memory=True,
        remove_unused_columns=False,
    )


final_config = (
    FALLBACK_CONFIG if oom_state_path.is_file() or active_config == FALLBACK_CONFIG
    else PRIMARY_CONFIG
)
final_summary_path = FINAL_DIR / "training_summary.json"
reuse_final_artifact = False
if final_summary_path.is_file():
    try:
        existing_final_summary = json.loads(
            final_summary_path.read_text(encoding="utf-8")
        )
        reuse_final_artifact = (
            existing_final_summary.get("signature") == final_signature(final_config)
            and validate_core_inventory(existing_final_summary.get("core_inventory", []))
        )
    except (OSError, ValueError, KeyError, json.JSONDecodeError):
        reuse_final_artifact = False

if reuse_final_artifact:
    final_training_record = existing_final_summary
    print("Reused validated frozen final local artifact.")
else:
    if FINAL_DIR.exists():
        safe_remove_directory(FINAL_DIR)
    if FINAL_TEMP_DIR.exists():
        safe_remove_directory(FINAL_TEMP_DIR)
    FINAL_DIR.mkdir(parents=True, exist_ok=True)
    FINAL_TEMP_DIR.mkdir(parents=True, exist_ok=True)

    set_seed(RANDOM_STATE)
    final_model = initialize_fresh_model()
    final_dataset = EncodedDevelopmentDataset(
        np.arange(EXPECTED_DEVELOPMENT_ROWS, dtype=np.int64)
    )
    final_trainer = Trainer(
        model=final_model,
        args=build_final_training_arguments(FINAL_TEMP_DIR, final_config),
        train_dataset=final_dataset,
        data_collator=data_collator,
    )

    clean_cuda()
    torch.cuda.reset_peak_memory_stats()
    final_wall_start = time.perf_counter()
    final_train_output = final_trainer.train()
    final_training_wall_seconds = time.perf_counter() - final_wall_start
    final_peak_allocated_mib = torch.cuda.max_memory_allocated() / (1024**2)
    final_peak_reserved_mib = torch.cuda.max_memory_reserved() / (1024**2)

    final_trainer.save_model(FINAL_DIR)
    tokenizer.save_pretrained(FINAL_DIR)
    core_inventory = directory_inventory(
        FINAL_DIR, excluded_names={"training_summary.json"}
    )
    final_training_record = {
        "signature": final_signature(final_config),
        "training_runtime_seconds": float(
            final_train_output.metrics.get(
                "train_runtime", final_training_wall_seconds
            )
        ),
        "training_wall_seconds": float(final_training_wall_seconds),
        "peak_allocated_mib": float(final_peak_allocated_mib),
        "peak_reserved_mib": float(final_peak_reserved_mib),
        "core_inventory": core_inventory,
    }
    final_summary_path.write_text(
        json.dumps(final_training_record, indent=2, sort_keys=True),
        encoding="utf-8",
    )

    del final_trainer, final_model, final_dataset, final_train_output
    clean_cuda()
    if FINAL_TEMP_DIR.exists():
        safe_remove_directory(FINAL_TEMP_DIR)

reloaded_model = DistilBertForSequenceClassification.from_pretrained(
    FINAL_DIR,
    local_files_only=True,
)
reloaded_tokenizer = AutoTokenizer.from_pretrained(
    FINAL_DIR,
    local_files_only=True,
    use_fast=True,
)
if reloaded_model.__class__.__name__ != MODEL_CLASS_NAME:
    raise RuntimeError("Reloaded final model class is incorrect.")
if reloaded_model.num_labels != len(CANONICAL_LABELS):
    raise RuntimeError("Reloaded final model does not have eight output labels.")
if reloaded_model.config.label2id != label2id:
    raise RuntimeError("Reloaded label2id mapping does not match the lock.")
if reloaded_model.config.id2label != id2label:
    raise RuntimeError("Reloaded id2label mapping does not match the lock.")
if reloaded_tokenizer.__class__.__name__ != "DistilBertTokenizerFast":
    raise RuntimeError("Reloaded tokenizer class is incorrect.")

all_model_parameters_finite = all(
    torch.isfinite(parameter).all().item()
    for parameter in reloaded_model.parameters()
)
if not all_model_parameters_finite:
    raise RuntimeError("Frozen final model contains non-finite parameters.")

synthetic_inputs = reloaded_tokenizer(
    "synthetic validation input",
    add_special_tokens=True,
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors="pt",
)
reloaded_model.eval()
with torch.no_grad():
    synthetic_logits = reloaded_model(**synthetic_inputs).logits
synthetic_output_shape = tuple(int(value) for value in synthetic_logits.shape)
synthetic_logits_finite = bool(torch.isfinite(synthetic_logits).all().item())
if synthetic_output_shape != (1, len(CANONICAL_LABELS)):
    raise RuntimeError(f"Unexpected synthetic output shape: {synthetic_output_shape}")
if not synthetic_logits_finite:
    raise RuntimeError("Reloaded final model produced non-finite synthetic logits.")

final_artifact_inventory = directory_inventory(FINAL_DIR)
final_artifact_size = sum(record["size_bytes"] for record in final_artifact_inventory)
final_core_model_record = next(
    (
        record
        for record in final_artifact_inventory
        if Path(record["path"]).name in {"model.safetensors", "pytorch_model.bin"}
    ),
    None,
)
if final_core_model_record is None:
    raise RuntimeError("Frozen final model weight file is missing.")

print("Frozen final development model:")
print(f"- Development rows used: {EXPECTED_DEVELOPMENT_ROWS:,}")
print(f"- Training epochs: {selected_final_epoch}")
print(f"- Configuration: {final_config['name']}")
print(
    f"- Training runtime: "
    f"{final_training_record['training_runtime_seconds'] / 60.0:.2f} minutes"
)
print(
    f"- Peak allocated / reserved GPU memory: "
    f"{final_training_record['peak_allocated_mib']:.1f} / "
    f"{final_training_record['peak_reserved_mib']:.1f} MiB"
)
print(f"- Local path: {FINAL_DIR.relative_to(PROJECT_ROOT).as_posix()}")
print(f"- Artifact files: {len(final_artifact_inventory)}")
print(f"- Total artifact size: {final_artifact_size:,} bytes")
print(
    f"- Model-weight SHA-256: {final_core_model_record['sha256']}"
)
print(f"- Reloaded synthetic output shape: {synthetic_output_shape}")
print("All saved model parameters finite: PASS")
print("Final model, label mappings, and tokenizer reload validation: PASS")

del reloaded_model, synthetic_inputs, synthetic_logits
clean_cuda()

Five fold-best integer epochs: [4, 2, 4, 4, 4]
Precommitted median-best-epoch selection: 4
The final epoch was recorded before final model training.


Reused validated frozen final local artifact.


Frozen final development model:
- Development rows used: 26,433
- Training epochs: 4
- Configuration: primary
- Training runtime: 349.84 minutes
- Peak allocated / reserved GPU memory: 1585.5 / 1738.0 MiB
- Local path: models/v2_distilbert_challenger/final
- Artifact files: 8
- Total artifact size: 268,806,032 bytes
- Model-weight SHA-256: e05900579f16e96d75df968cedb71b2b2fde3aae95f1bf73dbe7147306287c23
- Reloaded synthetic output shape: (1, 8)
All saved model parameters finite: PASS
Final model, label mappings, and tokenizer reload validation: PASS


## 8. Final Issue #40 validation

In [8]:
def git_ignored(path: Path) -> bool:
    return (
        subprocess.run(
            ["git", "check-ignore", "-q", path.relative_to(PROJECT_ROOT).as_posix()],
            cwd=PROJECT_ROOT,
            check=False,
        ).returncode
        == 0
    )


staged_paths = set(
    subprocess.check_output(
        ["git", "diff", "--cached", "--name-only"],
        cwd=PROJECT_ROOT,
        text=True,
    ).splitlines()
)
checkpoint_weight_files = list(FOLDS_DIR.glob("fold_*/checkpoints/**/model.safetensors"))
final_training_tmp_absent = not FINAL_TEMP_DIR.exists()

validation_checks = {
    "Pinned Version 2 environment": observed_environment == EXPECTED_ENVIRONMENT,
    "CUDA and locked GPU available": torch.cuda.is_available()
    and torch.cuda.get_device_name(0) == "NVIDIA GeForce GTX 1650",
    "Locked source path only": accessed_data_paths == {DATA_PATH.resolve()},
    "Source fingerprint": source_size == EXPECTED_SOURCE_SIZE
    and source_sha256 == EXPECTED_SOURCE_SHA256,
    "Corrected modeling rows": EXPECTED_MODELING_ROWS == 33_042,
    "Development rows": len(development_label_ids) == EXPECTED_DEVELOPMENT_ROWS,
    "Final-test boundary excluded": EXPECTED_FINAL_TEST_ROWS == 6_609
    and development_final_overlap == 0,
    "Five zero-overlap development folds": len(development_folds) == 5
    and all(row["Group overlap"] == 0 for row in fold_rows),
    "Every development row validates once": bool(np.all(validation_assignments == 1)),
    "Canonical eight-class mapping": label2id
    == {label: index for index, label in enumerate(CANONICAL_LABELS)},
    "Locked tokenizer and maximum length": tokenizer.__class__.__name__
    == "DistilBertTokenizerFast"
    and MAX_LENGTH == 256,
    "Five fold results completed": len(fold_results) == 5,
    "OOF assignment exactly once": bool(np.all(oof_assignment_counts == 1)),
    "OOF logits complete and finite": oof_logits.shape == (26_433, 8)
    and bool(np.isfinite(oof_logits).all()),
    "OOF predictions align with logits": bool(
        np.array_equal(oof_predictions, np.argmax(oof_logits, axis=1))
    ),
    "OOF artifact exists and is nonempty": oof_artifact_path.is_file()
    and oof_artifact_path.stat().st_size > 0,
    "Development confusion matrix exists": FIGURE_PATH.is_file()
    and FIGURE_PATH.stat().st_size > 0,
    "Median-best-epoch rule followed": selected_final_epoch
    == int(statistics.median(best_epochs)),
    "Final model trained on all development rows": final_training_record["signature"][
        "development_rows"
    ]
    == EXPECTED_DEVELOPMENT_ROWS,
    "Frozen model reload validation": synthetic_output_shape == (1, 8)
    and synthetic_logits_finite,
    "All saved model parameters finite": all_model_parameters_finite,
    "Final model mapping remains locked": reloaded_tokenizer.__class__.__name__
    == "DistilBertTokenizerFast",
    "Local artifacts are Git-ignored": git_ignored(oof_artifact_path)
    and git_ignored(final_core_model_record and PROJECT_ROOT / final_core_model_record["path"]),
    "No model or OOF artifact staged": not any(path.startswith("models/") for path in staged_paths),
    "Unnecessary fold checkpoints removed": len(checkpoint_weight_files) == 0,
    "Final temporary training directory removed": final_training_tmp_absent,
}

validation_table = pd.DataFrame(
    [(name, "PASS" if passed else "FAIL") for name, passed in validation_checks.items()],
    columns=["Check", "Result"],
)
failed_checks = [name for name, passed in validation_checks.items() if not passed]
print(validation_table.to_string(index=False))
print(
    f"Final Issue #40 validation: "
    f"{len(validation_checks) - len(failed_checks)}/{len(validation_checks)} checks passed"
)
if failed_checks:
    raise RuntimeError(f"Issue #40 validation failed: {failed_checks}")

print("Issue #40 development training and frozen-artifact gate: PASS")
print("No final internal-test or 2025/2026 tokenization, scoring, or evaluation occurred.")
print("No complaint narratives, IDs, hashes, row-level predictions, logits, or scores were displayed.")
print("Row-level OOF outputs and all model artifacts remain local and Git-ignored.")

                                      Check Result
               Pinned Version 2 environment   PASS
              CUDA and locked GPU available   PASS
                    Locked source path only   PASS
                         Source fingerprint   PASS
                    Corrected modeling rows   PASS
                           Development rows   PASS
               Final-test boundary excluded   PASS
        Five zero-overlap development folds   PASS
       Every development row validates once   PASS
              Canonical eight-class mapping   PASS
        Locked tokenizer and maximum length   PASS
                Five fold results completed   PASS
                OOF assignment exactly once   PASS
             OOF logits complete and finite   PASS
          OOF predictions align with logits   PASS
        OOF artifact exists and is nonempty   PASS
        Development confusion matrix exists   PASS
            Median-best-epoch rule followed   PASS
Final model trained on all deve